# PoinTr colored completion — Stage 0.5 (real geometry, oracle seg)

Swaps the **oracle geometry** of the local Stage-0 (`scripts/toy_part_color.py`) for
**real PoinTr-completed geometry**. Everything else stays oracle so PoinTr's geometry
error is the *only* new source:

* input: the synthetic **colour = part** toy (assumption true by construction, +σ noise).
* geometry: PoinTr fills the occluded region (pretrained ShapeNet55).
* **oracle seg**: each completed point's part = its **nearest GT point's part label**.
* colours compared: `baseline` (NN from visible) / `method` (part-mean over visible) /
  `ceiling` (part-mean over full GT = within-part noise floor).

> **Caveat:** PoinTr is trained on ShapeNet55 real shapes; the box-primitive toy is
> somewhat out-of-distribution, so geometry may be rough. That is fine for wiring the
> pipeline — swap `gt`/`gt_part` for a real chair (via `scripts/align_partseg.py` fusion)
> once ShapeNet-Part labels are downloaded, to get an in-distribution number.

Requires the Colab PoinTr setup at `/content/PoinTr` (same as `pointr_demo_colab.ipynb`).

In [ ]:
import os, sys, subprocess
POINTR = "/content/PoinTr"
assert os.path.isdir(POINTR), "PoinTr not found at /content/PoinTr - run your setup notebook first."
os.chdir(POINTR)
subprocess.run("pip install -q gdown open3d plotly", shell=True, check=True)
import torch
print("cwd:", os.getcwd(), "| CUDA:", torch.cuda.is_available(),
      "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
for ext in ["emd", "chamfer_dist", "cubic_feature_sampling"]:
    try:
        __import__({"emd":"emd","chamfer_dist":"chamfer","cubic_feature_sampling":"cubic_feature_sampling"}[ext])
    except Exception:
        print("building", ext, "...")
        subprocess.run(f"cd {POINTR}/extensions/{ext} && python setup.py install", shell=True, check=False)

In [ ]:
import os, subprocess
os.makedirs("ckpts", exist_ok=True)
CKPT = "ckpts/PoinTr_ShapeNet55.pth"
if not os.path.exists(CKPT) or os.path.getsize(CKPT) < 50e6:
    subprocess.run(f"gdown 1WzERLlbSwzGOBybzkjBrApwyVMTG00CJ -O {CKPT}", shell=True, check=True)
sz = os.path.getsize(CKPT) / 1e6
assert sz > 50, f"checkpoint too small ({sz:.1f} MB) - download failed"
print(f"checkpoint OK: {sz:.1f} MB")

### Toy input (colour = part). Mirrors `scripts/toy_part_color.py`.

In [ ]:
import numpy as np

CROP, SEED, COLOUR_NOISE = 0.5, 42, 0.03

PARTS = [
    ("seat",  (0.0, 0.90, 0.00), (2.0, 0.15, 2.0)),
    ("back",  (0.0, 1.65, -0.93), (2.0, 1.5, 0.15)),
    ("leg_a", (-0.9, 0.45, -0.9), (0.16, 0.9, 0.16)),
    ("leg_b", (0.9, 0.45, -0.9), (0.16, 0.9, 0.16)),
    ("leg_c", (-0.9, 0.45, 0.9), (0.16, 0.9, 0.16)),
    ("leg_d", (0.9, 0.45, 0.9), (0.16, 0.9, 0.16)),
]
NAMES = [p[0] for p in PARTS]
PALETTE = np.array([(0.85,0.20,0.20),(0.20,0.65,0.25),(0.20,0.35,0.80),
                    (0.20,0.35,0.80),(0.20,0.35,0.80),(0.20,0.35,0.80)])

def box_surface(center, size, n, rng):
    center=np.asarray(center,float); size=np.asarray(size,float); hx,hy,hz=size/2
    areas=np.array([hy*hz,hy*hz,hx*hz,hx*hz,hx*hy,hx*hy])*4
    cnt=np.random.default_rng(rng.integers(1<<30)).multinomial(n, areas/areas.sum()); pts=[]
    for f,k in enumerate(cnt):
        if k==0: continue
        u=rng.uniform(-1,1,k); v=rng.uniform(-1,1,k)
        if   f==0: p=np.stack([np.full(k,1.0),u,v],1)
        elif f==1: p=np.stack([np.full(k,-1.0),u,v],1)
        elif f==2: p=np.stack([u,np.full(k,1.0),v],1)
        elif f==3: p=np.stack([u,np.full(k,-1.0),v],1)
        elif f==4: p=np.stack([u,v,np.full(k,1.0)],1)
        else:      p=np.stack([u,v,np.full(k,-1.0)],1)
        pts.append(p*np.array([hx,hy,hz])+center)
    return np.concatenate(pts,0)

def build_toy(n=8192, seed=0, colour_noise=0.0):
    rng=np.random.default_rng(seed)
    sizes=np.array([s for _,_,s in PARTS],float)
    area=2*(sizes[:,0]*sizes[:,1]+sizes[:,1]*sizes[:,2]+sizes[:,0]*sizes[:,2])
    cnt=rng.multinomial(n, area/area.sum()); xyz,rgb,part=[],[],[]
    for pid,((name,c,s),k) in enumerate(zip(PARTS,cnt)):
        if k==0: continue
        p=box_surface(c,s,k,rng)
        col=PALETTE[pid]+(rng.normal(0,colour_noise,(k,3)) if colour_noise else 0.0)
        xyz.append(p); rgb.append(np.clip(col,0,1)); part.append(np.full(k,pid))
    return np.concatenate(xyz),np.concatenate(rgb),np.concatenate(part)

xyz, rgb, part = build_toy(8192, seed=SEED, colour_noise=COLOUR_NOISE)
gt = np.concatenate([xyz, rgb], 1).astype(np.float32); gt_part = part
print("toy GT:", gt.shape, "parts:", NAMES)

### Occlude (labels ride along) and hand PoinTr the partial geometry.

In [ ]:
def separate_idx(xyz, crop_ratio=0.5, seed=42):
    N=len(xyz); num_crop=int(round(N*crop_ratio))
    c=xyz.mean(0); xyzn=(xyz-c)/(np.linalg.norm(xyz-c,axis=1).max()+1e-12)
    rng=np.random.default_rng(seed); v=rng.standard_normal(3); center=v/np.linalg.norm(v)
    idx=np.argsort(np.linalg.norm(xyzn-center[None],axis=1))
    return idx[num_crop:], idx[:num_crop]

vis_idx, miss_idx = separate_idx(gt[:,:3], crop_ratio=CROP, seed=SEED)
partial = gt[vis_idx]; partial_part = gt_part[vis_idx]
print("partial:", partial.shape, "| missing:", len(miss_idx))

import open3d as o3d
os.makedirs("/content/demo/in", exist_ok=True); os.makedirs("/content/demo/out", exist_ok=True)
pin = o3d.geometry.PointCloud(); pin.points = o3d.utility.Vector3dVector(partial[:,:3].astype(np.float64))
o3d.io.write_point_cloud("/content/demo/in/partial.ply", pin)

In [ ]:
env = os.environ.copy(); env["PYTHONPATH"] = "/content/PoinTr:" + env.get("PYTHONPATH", "")
cmd = ("python tools/inference.py cfgs/ShapeNet55_models/PoinTr.yaml ckpts/PoinTr_ShapeNet55.pth "
       "--pc_root /content/demo/in --out_pc_root /content/demo/out --device cuda:0")
r = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd="/content/PoinTr", env=env)
print(r.stdout[-1500:]); print("STDERR:", r.stderr[-1500:])
completed = np.load("/content/demo/out/partial/fine.npy")   # (M,3) completed geometry
print("completed:", completed.shape)

### Colour the completed points 3 ways and score ΔE
`oracle seg` = nearest-GT part label; eval reference colour = nearest-GT rgb.

In [ ]:
from scipy.spatial import cKDTree

def srgb_to_lab(rgb):
    rgb=np.clip(rgb,0,1); lin=np.where(rgb>0.04045,((rgb+0.055)/1.055)**2.4,rgb/12.92)
    M=np.array([[0.4124,0.3576,0.1805],[0.2126,0.7152,0.0722],[0.0193,0.1192,0.9505]])
    xyz=(lin@M.T)/np.array([0.95047,1.0,1.08883]); d=6/29
    f=np.where(xyz>d**3,np.cbrt(xyz),xyz/(3*d**2)+4/29)
    return np.stack([116*f[:,1]-16,500*(f[:,0]-f[:,1]),200*(f[:,1]-f[:,2])],1)
def delta_e(a,b): return np.linalg.norm(srgb_to_lab(a)-srgb_to_lab(b),axis=1)
def part_means(rgb, part, P, fb):
    m=np.tile(fb,(P,1))
    for p in range(P):
        s=part==p
        if s.any(): m[p]=rgb[s].mean(0)
    return m

dg, gidx = cKDTree(gt[:,:3]).query(completed, k=1)
true_rgb  = gt[gidx, 3:6]        # eval reference colour at each completed point
comp_part = gt_part[gidx]        # ORACLE seg on completed = nearest-GT part label
_, pidx = cKDTree(partial[:,:3]).query(completed, k=1)

P = len(NAMES); glob = partial[:,3:6].mean(0)
colourings = {
    "baseline (NN)"            : partial[pidx, 3:6],
    "method (part-mean,vis)"   : part_means(partial[:,3:6], partial_part, P, glob)[comp_part],
    "ceiling (part-mean,full)" : part_means(gt[:,3:6], gt_part, P, glob)[comp_part],
}
d2,_ = cKDTree(completed).query(gt[:,:3], k=1)
chamfer = float((dg**2).mean() + (d2**2).mean())
print(f"Chamfer(L2,sym): {chamfer:.6f}   completed pts: {len(completed)}")
print(f"nearest-GT oracle-seg spread over parts: "
      + str({NAMES[p]: int((comp_part==p).sum()) for p in range(P)}))
print("\n--- mean ΔE(Lab) on completed region ---")
for name, col in colourings.items():
    print(f"  {name:26} {delta_e(col, true_rgb).mean():7.3f}")
method_rgb = colourings["method (part-mean,vis)"]

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
def sub(a,n=6000):
    return a if len(a)<=n else a[np.random.default_rng(0).choice(len(a),n,replace=False)]
def trace(xyz,rgb):
    a=np.concatenate([xyz,rgb],1); a=sub(a)
    c=["rgb(%d,%d,%d)"%(int(r*255),int(g*255),int(b*255)) for r,g,b in a[:,3:6]]
    return go.Scatter3d(x=a[:,0],y=a[:,1],z=a[:,2],mode="markers",marker=dict(size=1.8,color=c))
panels=[("partial (input)",partial[:,:3],partial[:,3:6]),
        ("baseline: NN",completed,colourings["baseline (NN)"]),
        ("method: part-mean",completed,method_rgb),
        ("GT",gt[:,:3],gt[:,3:6])]
fig=make_subplots(rows=1,cols=4,specs=[[{"type":"scene"}]*4],subplot_titles=[t for t,_,_ in panels])
for i,(_,x,r) in enumerate(panels,1): fig.add_trace(trace(x,r),1,i)
fig.update_layout(height=520,showlegend=False,margin=dict(l=0,r=0,t=30,b=0))
for i in range(1,5): fig.layout["scene" if i==1 else f"scene{i}"].aspectmode="data"
fig.show()